In [6]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


RANDOM_SEED = 52
rng = np.random.default_rng(RANDOM_SEED)


def make_age_group(age):
    if pd.isna(age):
        return "UNKNOWN"
    if age < 18:
        return "UNDER_18"
    if age <= 24:
        return "18_24"
    if age <= 34:
        return "25_34"
    if age <= 44:
        return "35_44"
    if age <= 54:
        return "45_54"
    if age <= 64:
        return "55_64"
    return "65_PLUS"


def prepare_income_features(df):
    df = df.copy()

    df = df.dropna(subset=["sex"])

    df["segment"] = df["segment"].fillna("UNKNOWN").astype(str)
    df["sex"] = df["sex"].astype(str)

    df["seniority_months"] = pd.to_numeric(df["seniority_months"], errors="coerce")
    df.loc[df["seniority_months"] < 0, "seniority_months"] = np.nan
    df = df.dropna(subset=["seniority_months"])
    df["seniority_months"] = df["seniority_months"].round().astype("float64")

    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df.loc[
        (df["age"] < 14) | (df["age"] > 95),
        "age"
    ] = np.nan
    df = df.dropna(subset=["age"])
    df["age"] = df["age"].round().astype("float64")

    df["age_group"] = df["age"].apply(make_age_group)

    prefixes = ("dep-", "card-", "rko-", "loan-", "srv-", "biz-")

    product_cols = [
        col for col in df.columns
        if col.startswith(prefixes)
    ]

    dep_cols = [col for col in product_cols if col.startswith("dep-")]
    card_cols = [col for col in product_cols if col.startswith("card-")]
    rko_cols = [col for col in product_cols if col.startswith("rko-")]
    loan_cols = [col for col in product_cols if col.startswith("loan-")]
    srv_cols = [col for col in product_cols if col.startswith("srv-")]
    biz_cols = [col for col in product_cols if col.startswith("biz-")]

    df["products_count"] = df[product_cols].sum(axis=1)

    df["dep_count"] = df[dep_cols].sum(axis=1) if dep_cols else 0
    df["card_count"] = df[card_cols].sum(axis=1) if card_cols else 0
    df["rko_count"] = df[rko_cols].sum(axis=1) if rko_cols else 0
    df["loan_count"] = df[loan_cols].sum(axis=1) if loan_cols else 0
    df["srv_count"] = df[srv_cols].sum(axis=1) if srv_cols else 0
    df["biz_count"] = df[biz_cols].sum(axis=1) if biz_cols else 0

    df["has_dep"] = (df["dep_count"] > 0).astype(int)
    df["has_card"] = (df["card_count"] > 0).astype(int)
    df["has_rko"] = (df["rko_count"] > 0).astype(int)
    df["has_loan"] = (df["loan_count"] > 0).astype(int)
    df["has_srv"] = (df["srv_count"] > 0).astype(int)
    df["has_biz"] = (df["biz_count"] > 0).astype(int)

    def product_profile(row):
        if row["products_count"] == 0:
            return "NO_PRODUCTS"

        profile = []

        if row["has_dep"]:
            profile.append("DEP")
        if row["has_card"]:
            profile.append("CARD")
        if row["has_rko"]:
            profile.append("RKO")
        if row["has_loan"]:
            profile.append("LOAN")
        if row["has_srv"]:
            profile.append("SRV")
        if row["has_biz"]:
            profile.append("BIZ")

        return "_".join(profile)

    df["product_profile"] = df.apply(product_profile, axis=1)

    return df, product_cols


def calculate_income_metrics(y_true_log, y_pred_log, title="Metrics"):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    medae = median_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmsle = mean_squared_error(np.log1p(y_true), np.log1p(y_pred)) ** 0.5
    r2_log = r2_score(y_true_log, y_pred_log)

    print(f"\n=== {title} ===")
    print(f"Rows: {len(y_true):,}")
    print(f"MAE: {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MedianAE: {medae:,.2f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"RMSLE: {rmsle:.4f}")
    print(f"R2 log-income: {r2_log:.4f}")

    return {
        "rows": len(y_true),
        "mae": mae,
        "rmse": rmse,
        "median_ae": medae,
        "mape": mape,
        "rmsle": rmsle,
        "r2_log": r2_log
    }


def generate_income_with_train_test(path):
    df = pd.read_csv(path)

    df, product_cols = prepare_income_features(df)

    known_mask = df["income"].notna() & (df["income"] > 0)
    known_df = df.loc[known_mask].copy()

    lower = known_df["income"].quantile(0.005)
    upper = known_df["income"].quantile(0.995)

    known_df = known_df[
        (known_df["income"] >= lower) &
        (known_df["income"] <= upper)
    ].copy()

    cat_features = [
        "sex",
        "segment",
        "age_group",
        "product_profile"
    ]

    num_features = [
        "age",
        "is_new_customer",
        "seniority_months",
        "products_count",
        "dep_count",
        "card_count",
        "rko_count",
        "loan_count",
        "srv_count",
        "biz_count",
        "has_dep",
        "has_card",
        "has_rko",
        "has_loan",
        "has_srv",
        "has_biz"
    ] + product_cols

    features = cat_features + num_features

    for col in num_features:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
        known_df[col] = pd.to_numeric(known_df[col], errors="coerce").astype("float64")

    X = known_df[features]
    y = np.log1p(known_df["income"])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_SEED
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                cat_features
            )
        ],
        remainder="passthrough",
        sparse_threshold=0
    )

    model = HistGradientBoostingRegressor(
        max_iter=1000,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=RANDOM_SEED
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    test_pred_log = pipeline.predict(X_test)

    deterministic_metrics = calculate_income_metrics(
        y_true_log=y_test,
        y_pred_log=test_pred_log,
        title="Deterministic model on test"
    )

    train_pred_log = pipeline.predict(X_train)

    train_residual_df = X_train.copy()
    train_residual_df["income_residual"] = y_train - train_pred_log

    train_residual_df["residual_group"] = (
        train_residual_df["segment"].astype(str) + "_" +
        train_residual_df["age_group"].astype(str) + "_" +
        train_residual_df["product_profile"].astype(str)
    )

    global_residuals = train_residual_df["income_residual"].values

    residuals_by_group = {
        group: values["income_residual"].values
        for group, values in train_residual_df.groupby("residual_group")
        if len(values) >= 30
    }

    residuals_by_segment_age = {
        group: values["income_residual"].values
        for group, values in train_residual_df.groupby(["segment", "age_group"])
        if len(values) >= 30
    }

    test_sampling_df = X_test.copy()
    test_sampling_df["residual_group"] = (
        test_sampling_df["segment"].astype(str) + "_" +
        test_sampling_df["age_group"].astype(str) + "_" +
        test_sampling_df["product_profile"].astype(str)
    )

    synthetic_test_log = []

    for pos, (_, row) in enumerate(test_sampling_df.iterrows()):
        base_prediction = test_pred_log[pos]
        group = row["residual_group"]

        if group in residuals_by_group:
            residual_pool = residuals_by_group[group]
        else:
            fallback_key = (row["segment"], row["age_group"])
            residual_pool = residuals_by_segment_age.get(
                fallback_key,
                global_residuals
            )

        sampled_residual = rng.choice(residual_pool)
        extra_noise = rng.normal(0, 0.03)

        synthetic_test_log.append(
            base_prediction + sampled_residual + extra_noise
        )

    synthetic_test_log = np.array(synthetic_test_log)

    synthetic_metrics = calculate_income_metrics(
        y_true_log=y_test,
        y_pred_log=synthetic_test_log,
        title="Synthetic income on test"
    )

    X_all = df[features]
    pred_all_log = pipeline.predict(X_all)

    df["residual_group"] = (
        df["segment"].astype(str) + "_" +
        df["age_group"].astype(str) + "_" +
        df["product_profile"].astype(str)
    )

    synthetic_log_incomes = []

    for pos, (_, row) in enumerate(df.iterrows()):
        base_prediction = pred_all_log[pos]
        group = row["residual_group"]

        if group in residuals_by_group:
            residual_pool = residuals_by_group[group]
        else:
            fallback_key = (row["segment"], row["age_group"])
            residual_pool = residuals_by_segment_age.get(
                fallback_key,
                global_residuals
            )

        sampled_residual = rng.choice(residual_pool)
        extra_noise = rng.normal(0, 0.03)

        synthetic_log_incomes.append(
            base_prediction + sampled_residual + extra_noise
        )

    income_generated = np.expm1(synthetic_log_incomes)
    income_generated = np.clip(income_generated, lower, upper)

    df["income_model_prediction"] = np.round(np.expm1(pred_all_log), 2)

    df["income_generated"] = np.nan

    missing_mask = df["income"].isna() | (df["income"] <= 0)

    df.loc[missing_mask, "income_generated"] = np.round(
        income_generated[missing_mask],
        2
    )

    df["income_filled"] = df["income"]

    df.loc[missing_mask, "income_filled"] = df.loc[
        missing_mask,
        "income_generated"
    ]

    df["income_was_generated"] = missing_mask.astype(int)

    df = df.drop(columns=["residual_group"], errors="ignore")

    metrics = {
        "deterministic_test": deterministic_metrics,
        "synthetic_test": synthetic_metrics
    }

    return df, pipeline, metrics


path = "raw/train_wide.csv"

df_income, income_model, income_metrics = generate_income_with_train_test(path)

df_income.to_csv(
    "processed/v1/train_wide_with_income_test.csv",
    index=False
)

print("\nSaved:")
print("processed/v1/train_wide_with_income_test.csv")


=== Deterministic model on test ===
Rows: 57,058
MAE: 56,696.55
RMSE: 87,358.19
MedianAE: 38,907.89
MAPE: 48.19%
RMSLE: 0.5567
R2 log-income: 0.0927

=== Synthetic income on test ===
Rows: 57,058
MAE: 83,230.58
RMSE: 120,932.07
MedianAE: 56,163.41
MAPE: 77.49%
RMSLE: 0.7869
R2 log-income: -0.8126

Saved:
processed/v1/train_wide_with_income_test.csv


In [2]:
import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# =========================
# CONFIG
# =========================

RAW_PATH = "raw/train_wide.csv"
OUTPUT_PATH = "processed/v1/train_wide_with_income_test.csv"

RANDOM_SEED = 52
rng = np.random.default_rng(RANDOM_SEED)

# Чем заполнять пропущенный income:
# "deterministic" — прогноз модели без шума, лучше для восстановления пропусков
# "synthetic" — прогноз + residual noise, лучше для реалистичного синтетического разброса
FILL_MODE = "deterministic"


# =========================
# FEATURE ENGINEERING
# =========================

def make_age_group(age):
    if pd.isna(age):
        return "UNKNOWN"
    if age < 18:
        return "UNDER_18"
    if age <= 24:
        return "18_24"
    if age <= 34:
        return "25_34"
    if age <= 44:
        return "35_44"
    if age <= 54:
        return "45_54"
    if age <= 64:
        return "55_64"
    return "65_PLUS"


def prepare_income_features(df):
    df = df.copy()

    # sex: удаляем редкие пропуски
    df = df.dropna(subset=["sex"])
    df["sex"] = df["sex"].astype(str)

    # segment: UNKNOWN вместо NaN
    df["segment"] = df["segment"].fillna("UNKNOWN").astype(str)

    # seniority_months
    df["seniority_months"] = pd.to_numeric(df["seniority_months"], errors="coerce")
    df.loc[df["seniority_months"] < 0, "seniority_months"] = np.nan
    df = df.dropna(subset=["seniority_months"])
    df["seniority_months"] = df["seniority_months"].round().astype("float64")

    # age
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df.loc[
        (df["age"] < 14) | (df["age"] > 95),
        "age"
    ] = np.nan
    df = df.dropna(subset=["age"])
    df["age"] = df["age"].round().astype("float64")

    df["age_group"] = df["age"].apply(make_age_group)

    # is_new_customer
    df["is_new_customer"] = pd.to_numeric(
        df["is_new_customer"],
        errors="coerce"
    ).fillna(0).astype("float64")

    # product columns
    prefixes = ("dep-", "card-", "rko-", "loan-", "srv-", "biz-")

    product_cols = [
        col for col in df.columns
        if col.startswith(prefixes)
    ]

    for col in product_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float64")

    dep_cols = [col for col in product_cols if col.startswith("dep-")]
    card_cols = [col for col in product_cols if col.startswith("card-")]
    rko_cols = [col for col in product_cols if col.startswith("rko-")]
    loan_cols = [col for col in product_cols if col.startswith("loan-")]
    srv_cols = [col for col in product_cols if col.startswith("srv-")]
    biz_cols = [col for col in product_cols if col.startswith("biz-")]

    # aggregate product features
    df["products_count"] = df[product_cols].sum(axis=1) if product_cols else 0

    df["dep_count"] = df[dep_cols].sum(axis=1) if dep_cols else 0
    df["card_count"] = df[card_cols].sum(axis=1) if card_cols else 0
    df["rko_count"] = df[rko_cols].sum(axis=1) if rko_cols else 0
    df["loan_count"] = df[loan_cols].sum(axis=1) if loan_cols else 0
    df["srv_count"] = df[srv_cols].sum(axis=1) if srv_cols else 0
    df["biz_count"] = df[biz_cols].sum(axis=1) if biz_cols else 0

    df["has_dep"] = (df["dep_count"] > 0).astype(int)
    df["has_card"] = (df["card_count"] > 0).astype(int)
    df["has_rko"] = (df["rko_count"] > 0).astype(int)
    df["has_loan"] = (df["loan_count"] > 0).astype(int)
    df["has_srv"] = (df["srv_count"] > 0).astype(int)
    df["has_biz"] = (df["biz_count"] > 0).astype(int)

    def product_profile(row):
        if row["products_count"] == 0:
            return "NO_PRODUCTS"

        profile = []

        if row["has_dep"]:
            profile.append("DEP")
        if row["has_card"]:
            profile.append("CARD")
        if row["has_rko"]:
            profile.append("RKO")
        if row["has_loan"]:
            profile.append("LOAN")
        if row["has_srv"]:
            profile.append("SRV")
        if row["has_biz"]:
            profile.append("BIZ")

        return "_".join(profile)

    df["product_profile"] = df.apply(product_profile, axis=1)

    return df, product_cols


# =========================
# METRICS
# =========================

def calculate_income_metrics(y_true_log, y_pred_log, title="Metrics"):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    medae = median_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmsle = mean_squared_error(np.log1p(y_true), np.log1p(y_pred)) ** 0.5
    r2_log = r2_score(y_true_log, y_pred_log)

    print(f"\n=== {title} ===")
    print(f"Rows: {len(y_true):,}")
    print(f"MAE: {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MedianAE: {medae:,.2f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"RMSLE: {rmsle:.4f}")
    print(f"R2 log-income: {r2_log:.4f}")

    return {
        "title": title,
        "rows": len(y_true),
        "mae": mae,
        "rmse": rmse,
        "median_ae": medae,
        "mape": mape,
        "rmsle": rmsle,
        "r2_log": r2_log
    }


def calculate_filtered_metrics(
    y_true_log,
    deterministic_pred_log,
    synthetic_pred_log,
    upper_quantile
):
    y_true = np.expm1(y_true_log)

    upper_limit = np.quantile(y_true, upper_quantile)
    keep_mask = y_true <= upper_limit

    print("\n" + "=" * 70)
    print(f"FILTER: remove top {round((1 - upper_quantile) * 100, 2)}% by real income")
    print(f"Income limit q{int(upper_quantile * 100)}: {upper_limit:,.2f}")
    print(f"Rows before: {len(y_true):,}")
    print(f"Rows after: {keep_mask.sum():,}")
    print(f"Rows removed: {(~keep_mask).sum():,}")
    print("=" * 70)

    det_metrics = calculate_income_metrics(
        y_true_log=np.array(y_true_log)[keep_mask],
        y_pred_log=np.array(deterministic_pred_log)[keep_mask],
        title=f"Deterministic model on filtered test q{int(upper_quantile * 100)}"
    )

    syn_metrics = calculate_income_metrics(
        y_true_log=np.array(y_true_log)[keep_mask],
        y_pred_log=np.array(synthetic_pred_log)[keep_mask],
        title=f"Synthetic income on filtered test q{int(upper_quantile * 100)}"
    )

    return det_metrics, syn_metrics


# =========================
# MAIN PIPELINE
# =========================

df = pd.read_csv(RAW_PATH)

print("Original rows:", len(df))

df, product_cols = prepare_income_features(df)

print("Rows after feature cleaning:", len(df))

known_mask = df["income"].notna() & (df["income"] > 0)
known_df = df.loc[known_mask].copy()

print("Rows with known income:", len(known_df))
print("Rows with missing income:", (~known_mask).sum())

# cut extreme income values only for training/evaluation
lower = known_df["income"].quantile(0.005)
upper = known_df["income"].quantile(0.995)

known_df = known_df[
    (known_df["income"] >= lower) &
    (known_df["income"] <= upper)
].copy()

print(f"Training income lower q0.5%: {lower:,.2f}")
print(f"Training income upper q99.5%: {upper:,.2f}")
print("Known income rows after extreme cut:", len(known_df))

cat_features = [
    "sex",
    "segment",
    "age_group",
    "product_profile"
]

num_features = [
    "age",
    "is_new_customer",
    "seniority_months",
    "products_count",
    "dep_count",
    "card_count",
    "rko_count",
    "loan_count",
    "srv_count",
    "biz_count",
    "has_dep",
    "has_card",
    "has_rko",
    "has_loan",
    "has_srv",
    "has_biz"
] + product_cols

features = cat_features + num_features

# force numeric dtype
for col in num_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    known_df[col] = pd.to_numeric(known_df[col], errors="coerce").astype("float64")

X = known_df[features]
y = np.log1p(known_df["income"])

# stratify by segment if possible
segment_counts = known_df["segment"].value_counts()
stratify_col = known_df["segment"] if segment_counts.min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=stratify_col
)

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            cat_features
        )
    ],
    remainder="passthrough",
    sparse_threshold=0
)

model = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=RANDOM_SEED
)

income_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# train
income_pipeline.fit(X_train, y_train)

# deterministic test prediction
test_pred_log = income_pipeline.predict(X_test)

deterministic_test_metrics = calculate_income_metrics(
    y_true_log=np.array(y_test),
    y_pred_log=np.array(test_pred_log),
    title="Deterministic model on full test"
)

# residuals from train only
train_pred_log = income_pipeline.predict(X_train)

train_residual_df = X_train.copy()
train_residual_df["income_residual"] = np.array(y_train) - train_pred_log

train_residual_df["residual_group"] = (
    train_residual_df["segment"].astype(str) + "_" +
    train_residual_df["age_group"].astype(str) + "_" +
    train_residual_df["product_profile"].astype(str)
)

global_residuals = train_residual_df["income_residual"].values

residuals_by_group = {
    group: values["income_residual"].values
    for group, values in train_residual_df.groupby("residual_group")
    if len(values) >= 30
}

residuals_by_segment_age = {
    group: values["income_residual"].values
    for group, values in train_residual_df.groupby(["segment", "age_group"])
    if len(values) >= 30
}

# synthetic test prediction
test_sampling_df = X_test.copy()
test_sampling_df["residual_group"] = (
    test_sampling_df["segment"].astype(str) + "_" +
    test_sampling_df["age_group"].astype(str) + "_" +
    test_sampling_df["product_profile"].astype(str)
)

synthetic_test_log = []

for pos, (_, row) in enumerate(test_sampling_df.iterrows()):
    base_prediction = test_pred_log[pos]
    group = row["residual_group"]

    if group in residuals_by_group:
        residual_pool = residuals_by_group[group]
    else:
        fallback_key = (row["segment"], row["age_group"])
        residual_pool = residuals_by_segment_age.get(
            fallback_key,
            global_residuals
        )

    sampled_residual = rng.choice(residual_pool)

    # можно уменьшать, если synthetic слишком шумный
    residual_strength = 1.0
    extra_noise_sigma = 0.03

    extra_noise = rng.normal(0, extra_noise_sigma)

    synthetic_test_log.append(
        base_prediction + residual_strength * sampled_residual + extra_noise
    )

synthetic_test_log = np.array(synthetic_test_log)

synthetic_test_metrics = calculate_income_metrics(
    y_true_log=np.array(y_test),
    y_pred_log=synthetic_test_log,
    title="Synthetic income on full test"
)

# filtered metrics: cut rich clients from test
filtered_metrics = {}

for q in [0.90, 0.95, 0.99]:
    det_m, syn_m = calculate_filtered_metrics(
        y_true_log=np.array(y_test),
        deterministic_pred_log=np.array(test_pred_log),
        synthetic_pred_log=np.array(synthetic_test_log),
        upper_quantile=q
    )

    filtered_metrics[f"deterministic_q{int(q * 100)}"] = det_m
    filtered_metrics[f"synthetic_q{int(q * 100)}"] = syn_m


# =========================
# GENERATE INCOME FOR FULL DATASET
# =========================

X_all = df[features]
pred_all_log = income_pipeline.predict(X_all)

df["income_model_prediction"] = np.round(np.expm1(pred_all_log), 2)

df["residual_group"] = (
    df["segment"].astype(str) + "_" +
    df["age_group"].astype(str) + "_" +
    df["product_profile"].astype(str)
)

synthetic_all_log = []

for pos, (_, row) in enumerate(df.iterrows()):
    base_prediction = pred_all_log[pos]
    group = row["residual_group"]

    if group in residuals_by_group:
        residual_pool = residuals_by_group[group]
    else:
        fallback_key = (row["segment"], row["age_group"])
        residual_pool = residuals_by_segment_age.get(
            fallback_key,
            global_residuals
        )

    sampled_residual = rng.choice(residual_pool)

    residual_strength = 1.0
    extra_noise_sigma = 0.03

    extra_noise = rng.normal(0, extra_noise_sigma)

    synthetic_all_log.append(
        base_prediction + residual_strength * sampled_residual + extra_noise
    )

income_synthetic_all = np.expm1(synthetic_all_log)
income_synthetic_all = np.clip(income_synthetic_all, lower, upper)

df["income_synthetic_prediction"] = np.round(income_synthetic_all, 2)

missing_mask = df["income"].isna() | (df["income"] <= 0)

df["income_generated"] = np.nan

if FILL_MODE == "deterministic":
    df.loc[missing_mask, "income_generated"] = df.loc[
        missing_mask,
        "income_model_prediction"
    ]
elif FILL_MODE == "synthetic":
    df.loc[missing_mask, "income_generated"] = df.loc[
        missing_mask,
        "income_synthetic_prediction"
    ]
else:
    raise ValueError("FILL_MODE must be either 'deterministic' or 'synthetic'.")

df["income_filled"] = df["income"]

df.loc[missing_mask, "income_filled"] = df.loc[
    missing_mask,
    "income_generated"
]

df["income_was_generated"] = missing_mask.astype(int)

df = df.drop(columns=["residual_group"], errors="ignore")

# save
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:")
print(OUTPUT_PATH)

print("\nFill mode:", FILL_MODE)
print("Generated income rows:", int(df["income_was_generated"].sum()))

# metrics as dataframe
all_metrics = {
    "deterministic_full_test": deterministic_test_metrics,
    "synthetic_full_test": synthetic_test_metrics,
    **filtered_metrics
}

metrics_df = pd.DataFrame(all_metrics).T

print("\nMetrics summary:")
display(metrics_df)

Original rows: 376141
Rows after feature cleaning: 372501
Rows with known income: 288171
Rows with missing income: 84330
Training income lower q0.5%: 23,917.68
Training income upper q99.5%: 727,150.70
Known income rows after extreme cut: 285289

Train rows: 228231
Test rows: 57058

=== Deterministic model on full test ===
Rows: 57,058
MAE: 56,859.43
RMSE: 88,168.98
MedianAE: 38,742.66
MAPE: 48.10%
RMSLE: 0.5575
R2 log-income: 0.0943

=== Synthetic income on full test ===
Rows: 57,058
MAE: 83,649.78
RMSE: 121,718.37
MedianAE: 56,397.09
MAPE: 77.87%
RMSLE: 0.7909
R2 log-income: -0.8226

FILTER: remove top 10.0% by real income
Income limit q90: 236,366.01
Rows before: 57,058
Rows after: 51,352
Rows removed: 5,706

=== Deterministic model on filtered test q90 ===
Rows: 51,352
MAE: 39,270.43
RMSE: 48,182.88
MedianAE: 34,612.87
MAPE: 46.67%
RMSLE: 0.4758
R2 log-income: 0.0139

=== Synthetic income on filtered test q90 ===
Rows: 51,352
MAE: 69,880.20
RMSE: 100,571.56
MedianAE: 50,087.77
MAPE:

,title,rows,mae,rmse,median_ae,mape,rmsle,r2_log
deterministic_full_test,Deterministic model on full test,57058,56859.433597,88168.983412,38742.659197,48.096328,0.557515,0.09426
synthetic_full_test,Synthetic income on full test,57058,83649.778263,121718.373048,56397.087385,77.872288,0.790852,-0.822561
deterministic_q90,Deterministic model on filtered test q90,51352,39270.428229,48182.879248,34612.871807,46.669131,0.475773,0.013904
synthetic_q90,Synthetic income on filtered test q90,51352,69880.203199,100571.560874,50087.768829,79.957864,0.734147,-1.347933
deterministic_q95,Deterministic model on filtered test q95,54205,44691.856604,57515.689427,36666.295726,47.021708,0.498308,0.069015
synthetic_q95,Synthetic income on filtered test q95,54205,73646.626929,104196.267556,53002.456335,78.54827,0.749531,-1.106335
deterministic_q99,Deterministic model on filtered test q99,56487,52907.932467,75993.560944,38346.28166,47.793601,0.538095,0.092987
synthetic_q99,Synthetic income on filtered test q99,56487,80148.095848,113861.305181,55617.425442,77.901869,0.776507,-0.888804


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# =========================
# CONFIG
# =========================

RAW_PATH = "raw/train_wide.csv"
OUTPUT_PATH = "processed/v1/train_wide_with_income_test_cut_high_income.csv"

RANDOM_SEED = 52
rng = np.random.default_rng(RANDOM_SEED)

# Удаляем из обучения и теста клиентов с income выше этого квантиля.
# 0.95 = убрать верхние 5% по доходу.
TRAIN_INCOME_UPPER_QUANTILE = 0.9

# Нижний срез можно оставить очень мягким.
TRAIN_INCOME_LOWER_QUANTILE = 0.005

# Чем заполнять пропущенный income:
# "deterministic" — прогноз модели без шума, лучше для восстановления пропусков
# "synthetic" — прогноз + residual noise, лучше для реалистичного синтетического разброса
FILL_MODE = "deterministic"

# Настройки synthetic-шума
RESIDUAL_STRENGTH = 1.0
EXTRA_NOISE_SIGMA = 0.03


# =========================
# FEATURE ENGINEERING
# =========================

def make_age_group(age):
    if pd.isna(age):
        return "UNKNOWN"
    if age < 18:
        return "UNDER_18"
    if age <= 24:
        return "18_24"
    if age <= 34:
        return "25_34"
    if age <= 44:
        return "35_44"
    if age <= 54:
        return "45_54"
    if age <= 64:
        return "55_64"
    return "65_PLUS"


def prepare_income_features(df):
    df = df.copy()

    # sex: удаляем редкие пропуски
    df = df.dropna(subset=["sex"])
    df["sex"] = df["sex"].astype(str)

    # segment: UNKNOWN вместо NaN
    df["segment"] = df["segment"].fillna("UNKNOWN").astype(str)

    # seniority_months
    df["seniority_months"] = pd.to_numeric(df["seniority_months"], errors="coerce")
    df.loc[df["seniority_months"] < 0, "seniority_months"] = np.nan
    df = df.dropna(subset=["seniority_months"])
    df["seniority_months"] = df["seniority_months"].round().astype("float64")

    # age
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df.loc[
        (df["age"] < 14) | (df["age"] > 95),
        "age"
    ] = np.nan
    df = df.dropna(subset=["age"])
    df["age"] = df["age"].round().astype("float64")

    df["age_group"] = df["age"].apply(make_age_group)

    # is_new_customer
    df["is_new_customer"] = pd.to_numeric(
        df["is_new_customer"],
        errors="coerce"
    ).fillna(0).astype("float64")

    # product columns
    prefixes = ("dep-", "card-", "rko-", "loan-", "srv-", "biz-")

    product_cols = [
        col for col in df.columns
        if col.startswith(prefixes)
    ]

    for col in product_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float64")

    dep_cols = [col for col in product_cols if col.startswith("dep-")]
    card_cols = [col for col in product_cols if col.startswith("card-")]
    rko_cols = [col for col in product_cols if col.startswith("rko-")]
    loan_cols = [col for col in product_cols if col.startswith("loan-")]
    srv_cols = [col for col in product_cols if col.startswith("srv-")]
    biz_cols = [col for col in product_cols if col.startswith("biz-")]

    # aggregate product features
    df["products_count"] = df[product_cols].sum(axis=1) if product_cols else 0

    df["dep_count"] = df[dep_cols].sum(axis=1) if dep_cols else 0
    df["card_count"] = df[card_cols].sum(axis=1) if card_cols else 0
    df["rko_count"] = df[rko_cols].sum(axis=1) if rko_cols else 0
    df["loan_count"] = df[loan_cols].sum(axis=1) if loan_cols else 0
    df["srv_count"] = df[srv_cols].sum(axis=1) if srv_cols else 0
    df["biz_count"] = df[biz_cols].sum(axis=1) if biz_cols else 0

    df["has_dep"] = (df["dep_count"] > 0).astype(int)
    df["has_card"] = (df["card_count"] > 0).astype(int)
    df["has_rko"] = (df["rko_count"] > 0).astype(int)
    df["has_loan"] = (df["loan_count"] > 0).astype(int)
    df["has_srv"] = (df["srv_count"] > 0).astype(int)
    df["has_biz"] = (df["biz_count"] > 0).astype(int)

    def product_profile(row):
        if row["products_count"] == 0:
            return "NO_PRODUCTS"

        profile = []

        if row["has_dep"]:
            profile.append("DEP")
        if row["has_card"]:
            profile.append("CARD")
        if row["has_rko"]:
            profile.append("RKO")
        if row["has_loan"]:
            profile.append("LOAN")
        if row["has_srv"]:
            profile.append("SRV")
        if row["has_biz"]:
            profile.append("BIZ")

        return "_".join(profile)

    df["product_profile"] = df.apply(product_profile, axis=1)

    return df, product_cols


# =========================
# METRICS
# =========================

def calculate_income_metrics(y_true_log, y_pred_log, title="Metrics"):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    medae = median_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmsle = mean_squared_error(np.log1p(y_true), np.log1p(y_pred)) ** 0.5
    r2_log = r2_score(y_true_log, y_pred_log)

    print(f"\n=== {title} ===")
    print(f"Rows: {len(y_true):,}")
    print(f"MAE: {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MedianAE: {medae:,.2f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"RMSLE: {rmsle:.4f}")
    print(f"R2 log-income: {r2_log:.4f}")

    return {
        "title": title,
        "rows": len(y_true),
        "mae": mae,
        "rmse": rmse,
        "median_ae": medae,
        "mape": mape,
        "rmsle": rmsle,
        "r2_log": r2_log
    }


def calculate_segment_metrics(X_test, y_true_log, y_pred_log, title="Segment metrics"):
    temp = X_test.copy()
    temp["y_true"] = np.expm1(y_true_log)
    temp["y_pred"] = np.expm1(y_pred_log)

    rows = []

    for segment, g in temp.groupby("segment"):
        if len(g) < 30:
            continue

        mae = mean_absolute_error(g["y_true"], g["y_pred"])
        rmse = mean_squared_error(g["y_true"], g["y_pred"]) ** 0.5
        medae = median_absolute_error(g["y_true"], g["y_pred"])
        mape = np.mean(np.abs((g["y_true"] - g["y_pred"]) / g["y_true"])) * 100
        rmsle = mean_squared_error(
            np.log1p(g["y_true"]),
            np.log1p(g["y_pred"])
        ) ** 0.5

        rows.append({
            "segment": segment,
            "rows": len(g),
            "mae": mae,
            "rmse": rmse,
            "median_ae": medae,
            "mape": mape,
            "rmsle": rmsle
        })

    result = pd.DataFrame(rows).sort_values("mae", ascending=False)

    print(f"\n=== {title} ===")
    display(result)

    return result


# =========================
# MAIN PIPELINE
# =========================

df = pd.read_csv(RAW_PATH)

print("Original rows:", len(df))

df, product_cols = prepare_income_features(df)

print("Rows after feature cleaning:", len(df))

known_mask = df["income"].notna() & (df["income"] > 0)
known_df_raw = df.loc[known_mask].copy()

print("Rows with known income before high-income cut:", len(known_df_raw))
print("Rows with missing income:", int((~known_mask).sum()))

# =========================
# CUT HIGH INCOME BEFORE TRAIN/TEST
# =========================

lower = known_df_raw["income"].quantile(TRAIN_INCOME_LOWER_QUANTILE)
upper = known_df_raw["income"].quantile(TRAIN_INCOME_UPPER_QUANTILE)

known_df = known_df_raw[
    (known_df_raw["income"] >= lower) &
    (known_df_raw["income"] <= upper)
].copy()

removed_known_rows = len(known_df_raw) - len(known_df)

print("\nIncome cut before train/test:")
print(f"Lower quantile q{TRAIN_INCOME_LOWER_QUANTILE}: {lower:,.2f}")
print(f"Upper quantile q{TRAIN_INCOME_UPPER_QUANTILE}: {upper:,.2f}")
print("Known rows after income cut:", len(known_df))
print("Known rows removed:", removed_known_rows)
print(f"Removed share: {removed_known_rows / len(known_df_raw) * 100:.2f}%")

cat_features = [
    "sex",
    "segment",
    "age_group",
    "product_profile"
]

num_features = [
    "age",
    "is_new_customer",
    "seniority_months",
    "products_count",
    "dep_count",
    "card_count",
    "rko_count",
    "loan_count",
    "srv_count",
    "biz_count",
    "has_dep",
    "has_card",
    "has_rko",
    "has_loan",
    "has_srv",
    "has_biz"
] + product_cols

features = cat_features + num_features

# force numeric dtype
for col in num_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    known_df[col] = pd.to_numeric(known_df[col], errors="coerce").astype("float64")

X = known_df[features]
y = np.log1p(known_df["income"])

# stratify by segment if possible
segment_counts = known_df["segment"].value_counts()
stratify_col = known_df["segment"] if segment_counts.min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=stratify_col
)

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))
print(f"Test max real income after cut: {np.expm1(y_test).max():,.2f}")

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            cat_features
        )
    ],
    remainder="passthrough",
    sparse_threshold=0
)

model = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=RANDOM_SEED
)

income_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# train
income_pipeline.fit(X_train, y_train)

# =========================
# TRAIN METRICS
# =========================

train_pred_log = income_pipeline.predict(X_train)

deterministic_train_metrics = calculate_income_metrics(
    y_true_log=np.array(y_train),
    y_pred_log=np.array(train_pred_log),
    title=f"Deterministic model on train after removing high income > q{int(TRAIN_INCOME_UPPER_QUANTILE * 100)}"
)

train_segment_metrics = calculate_segment_metrics(
    X_test=X_train,
    y_true_log=np.array(y_train),
    y_pred_log=np.array(train_pred_log),
    title="Deterministic train metrics by segment"
)

# deterministic test prediction
test_pred_log = income_pipeline.predict(X_test)

deterministic_test_metrics = calculate_income_metrics(
    y_true_log=np.array(y_test),
    y_pred_log=np.array(test_pred_log),
    title=f"Deterministic model on test after removing high income > q{int(TRAIN_INCOME_UPPER_QUANTILE * 100)}"
)

segment_metrics = calculate_segment_metrics(
    X_test=X_test,
    y_true_log=np.array(y_test),
    y_pred_log=np.array(test_pred_log),
    title="Deterministic test metrics by segment"
)


train_residual_df = X_train.copy()
train_residual_df["income_residual"] = np.array(y_train) - train_pred_log

train_residual_df["residual_group"] = (
    train_residual_df["segment"].astype(str) + "_" +
    train_residual_df["age_group"].astype(str) + "_" +
    train_residual_df["product_profile"].astype(str)
)

global_residuals = train_residual_df["income_residual"].values

residuals_by_group = {
    group: values["income_residual"].values
    for group, values in train_residual_df.groupby("residual_group")
    if len(values) >= 30
}

residuals_by_segment_age = {
    group: values["income_residual"].values
    for group, values in train_residual_df.groupby(["segment", "age_group"])
    if len(values) >= 30
}

# synthetic test prediction
test_sampling_df = X_test.copy()
test_sampling_df["residual_group"] = (
    test_sampling_df["segment"].astype(str) + "_" +
    test_sampling_df["age_group"].astype(str) + "_" +
    test_sampling_df["product_profile"].astype(str)
)

synthetic_test_log = []

for pos, (_, row) in enumerate(test_sampling_df.iterrows()):
    base_prediction = test_pred_log[pos]
    group = row["residual_group"]

    if group in residuals_by_group:
        residual_pool = residuals_by_group[group]
    else:
        fallback_key = (row["segment"], row["age_group"])
        residual_pool = residuals_by_segment_age.get(
            fallback_key,
            global_residuals
        )

    sampled_residual = rng.choice(residual_pool)
    extra_noise = rng.normal(0, EXTRA_NOISE_SIGMA)

    synthetic_test_log.append(
        base_prediction + RESIDUAL_STRENGTH * sampled_residual + extra_noise
    )

synthetic_test_log = np.array(synthetic_test_log)

synthetic_test_metrics = calculate_income_metrics(
    y_true_log=np.array(y_test),
    y_pred_log=synthetic_test_log,
    title=f"Synthetic income on test after removing high income > q{int(TRAIN_INCOME_UPPER_QUANTILE * 100)}"
)


# =========================
# GENERATE INCOME FOR FULL DATASET
# =========================

X_all = df[features]
pred_all_log = income_pipeline.predict(X_all)

df["income_model_prediction"] = np.round(np.expm1(pred_all_log), 2)

df["residual_group"] = (
    df["segment"].astype(str) + "_" +
    df["age_group"].astype(str) + "_" +
    df["product_profile"].astype(str)
)

synthetic_all_log = []

for pos, (_, row) in enumerate(df.iterrows()):
    base_prediction = pred_all_log[pos]
    group = row["residual_group"]

    if group in residuals_by_group:
        residual_pool = residuals_by_group[group]
    else:
        fallback_key = (row["segment"], row["age_group"])
        residual_pool = residuals_by_segment_age.get(
            fallback_key,
            global_residuals
        )

    sampled_residual = rng.choice(residual_pool)
    extra_noise = rng.normal(0, EXTRA_NOISE_SIGMA)

    synthetic_all_log.append(
        base_prediction + RESIDUAL_STRENGTH * sampled_residual + extra_noise
    )

income_synthetic_all = np.expm1(synthetic_all_log)

# Так как модель обучалась без верхнего хвоста, синтетику тоже ограничиваем тем же upper.
income_synthetic_all = np.clip(income_synthetic_all, lower, upper)

df["income_synthetic_prediction"] = np.round(income_synthetic_all, 2)

missing_mask = df["income"].isna() | (df["income"] <= 0)

df["income_generated"] = np.nan

if FILL_MODE == "deterministic":
    df.loc[missing_mask, "income_generated"] = df.loc[
        missing_mask,
        "income_model_prediction"
    ]
elif FILL_MODE == "synthetic":
    df.loc[missing_mask, "income_generated"] = df.loc[
        missing_mask,
        "income_synthetic_prediction"
    ]
else:
    raise ValueError("FILL_MODE must be either 'deterministic' or 'synthetic'.")

df["income_filled"] = df["income"]

df.loc[missing_mask, "income_filled"] = df.loc[
    missing_mask,
    "income_generated"
]

df["income_was_generated"] = missing_mask.astype(int)

df = df.drop(columns=["residual_group"], errors="ignore")

# save
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:")
print(OUTPUT_PATH)

print("\nFill mode:", FILL_MODE)
print("Generated income rows:", int(df["income_was_generated"].sum()))

all_metrics = {
    "deterministic_train_after_high_income_cut": deterministic_train_metrics,
    "deterministic_test_after_high_income_cut": deterministic_test_metrics,
    "synthetic_test_after_high_income_cut": synthetic_test_metrics
}

metrics_df = pd.DataFrame(all_metrics).T

print("\nMetrics summary:")
display(metrics_df)

Original rows: 376141
Rows after feature cleaning: 372501
Rows with known income before high-income cut: 288171
Rows with missing income: 84330

Income cut before train/test:
Lower quantile q0.005: 23,917.68
Upper quantile q0.85: 206,312.55
Known rows after income cut: 243504
Known rows removed: 44667
Removed share: 15.50%

Train rows: 194803
Test rows: 48701
Test max real income after cut: 206,306.91

=== Deterministic model on train after removing high income > q85 ===
Rows: 194,803
MAE: 34,625.06
RMSE: 43,096.55
MedianAE: 29,516.83
MAPE: 39.09%
RMSLE: 0.4370
R2 log-income: 0.0667

=== Deterministic train metrics by segment ===


,segment,rows,mae,rmse,median_ae,mape,rmsle
2,VIP,16655,36945.605384,44402.643915,33758.144748,37.145030,0.411442
0,INDIVIDUALS,128790,34997.688880,43349.684551,30052.613091,39.023077,0.434774
1,STUDENTS,49346,32869.113772,41971.760499,26982.874912,39.926388,0.450858



=== Deterministic model on test after removing high income > q85 ===
Rows: 48,701
MAE: 34,662.49
RMSE: 43,112.21
MedianAE: 29,571.87
MAPE: 39.31%
RMSLE: 0.4388
R2 log-income: 0.0625

=== Deterministic test metrics by segment ===


,segment,rows,mae,rmse,median_ae,mape,rmsle
2,VIP,4164,37135.426912,44720.687980,33395.243388,37.559235,0.415475
0,INDIVIDUALS,32198,35055.943541,43415.268524,30190.924964,39.144799,0.436128
1,STUDENTS,12336,32797.175973,41741.332698,26938.048924,40.323483,0.452994



=== Synthetic income on test after removing high income > q85 ===
Rows: 48,701
MAE: 47,936.83
RMSE: 59,961.45
MedianAE: 40,351.40
MAPE: 56.59%
RMSLE: 0.6213
R2 log-income: -0.8796

Saved:
processed/v1/train_wide_with_income_test_cut_high_income.csv

Fill mode: deterministic
Generated income rows: 84330

Metrics summary:


,title,rows,mae,rmse,median_ae,mape,rmsle,r2_log
deterministic_train_after_high_income_cut,Deterministic model on train after removing hi...,194803,34625.058625,43096.547989,29516.83213,39.090625,0.436973,0.066688
deterministic_test_after_high_income_cut,Deterministic model on test after removing hig...,48701,34662.49458,43112.207685,29571.867553,39.311286,0.438771,0.062516
synthetic_test_after_high_income_cut,Synthetic income on test after removing high i...,48701,47936.83196,59961.450049,40351.39622,56.586912,0.621282,-0.879604
